In [2]:
import pandas as pd
from lxml import etree
from generate_xml import load_data

In [3]:
data,metadata = load_data('../../zaebuc_written/ZAEBUC-v2.0_release/')
data.head()

/Users/f/Library/CloudStorage/SynologyDrive-ba3sasah/camelLab/zaebuc/corpus_app/src/generate_xml.py:146: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  en = pd.read_csv(f'{datadir}corrected.analyzed_en.tsv',sep='\t',index_col=[0,2,1])


word flag auto_tokenization auto_pos  \
doc_id         Line_Index idx                                                 
en-2019-116710 1.0        1    Developments  NaN      Developments     NOUN   
                          2              in  NaN                in      ADP   
                          3             the  NaN               the      DET   
                          4             UAE  NaN               UAE    PROPN   
                          5               ,  NaN                 ,    PUNCT   

                                auto_lemma manual_tokenization manual_pos  \
doc_id         Line_Index idx                                               
en-2019-116710 1.0        1    development        Developments       NOUN   
                          2             in                  in        ADP   
                          3            the                 the        DET   
                          4            UAE                 UAE      PROPN   
                          5              ,                   ,      PUNCT   

                              manual_lemma comment manual_diacritized_lemma  \
doc_id         Line_Index idx                                                 
en-2019-116710 1.0        1    development     NaN                      NaN   
                          2             in     NaN                      NaN   
                          3            the     NaN                      NaN   
                          4            UAE     NaN                      NaN   
                          5              ,     NaN                      NaN   

                              gloss core_pgn pron_pgn  
doc_id         Line_Index idx                          
en-2019-116710 1.0        1     NaN      NaN      NaN  
                          2     NaN      NaN      NaN  
                          3     NaN      NaN      NaN  
                          4     NaN      NaN      NaN  
                          5     NaN      NaN      NaN

In [4]:
from lxml import etree
import re

def write_xml(data,metadata,out_path="../data/zaebuc_written.xml",sample=None):
    if sample:
        metadata = metadata.sample(sample)
                
    namespaces = {'xml': 'http://www.w3.org/XML/1998/namespace'}
    corpus = etree.Element("corpus")

    # loop through docs
    for doc_id in metadata.index[:]:
    # for doc_id in ['ar-2021-X32635', 'ar-2021-X32635']:
        
        # doc metadata
        doc = metadata.loc[doc_id].to_frame().drop('text').T.reset_index(names='idx')
        doc['textDirection'] = doc['language'].map(lambda x: 'rtl' if x=='Arabic' else 'ltr')        
        doc = doc.to_xml(root_name='corpus',row_name='doc',attr_cols=doc.columns.to_list(),xml_declaration=False,index=False)
        doc = etree.fromstring(doc)

        linesidxs = data.loc[doc_id].index.get_level_values(0).drop_duplicates()
        
        #loop through lines "Line_Index" (think like sentences)
        for lineidx in linesidxs:        
            line = etree.Element('sent')
            line.attrib['idx'] = str(int(lineidx))        
            doc[0].append(line)
            
            words = data.loc[(doc_id,lineidx)][:]            
            
            # loop through words
            for idx in words.index:
                
                # punct handled as separate tag
                if words.loc[idx,'manual_pos'] == 'PUNCT':
                    word = etree.Element('punct')
                    word.attrib[f'{{{namespaces["xml"]}}}id'] = f'w.wx.{idx}'            
                    word.text = words.loc[idx,'word']
                    line.append(word)
                    continue
                else:                
                    word = etree.Element('word')
                
                #set idx and xml:id attributes
                word.attrib['idx'] = str(idx)            
                word.attrib[f'{{{namespaces["xml"]}}}id'] = f'w.wx.{idx}'    

                # set english gloss as english lemma
                if 'en' in doc_id:
                    gloss_value = words.loc[idx,'manual_lemma'].replace('+','')
                    gloss_element = etree.Element('gloss')
                    gloss_element.set('class',gloss_value)
                    word.append(gloss_element)
                
                # loop through analysis columns (extracted.analyzed sheet), 
                #  set them as class attributes, except for word
                for analysisid in words.columns:
                    value = words.loc[idx, analysisid]
                    
                    if pd.isna(value):  # skip nans
                        continue 
                    
                    # words go under text tag: <word><text>asdfa</text></word> 
                    if analysisid == 'word':
                        analysis = etree.Element('text')
                        analysis.text = value
                    
                    # split gloss into gloss_search elements
                    elif analysisid == 'gloss':                        
                        full_gloss_element = etree.Element("gloss")                                            
                        full_gloss_element.set('class', value)
                        word.append(full_gloss_element)
                        glosses = re.split(r',|;', value)
                        
                        for gloss in set(glosses):
                            gloss = gloss.strip()
                            gloss_element = etree.Element("gloss_search")
                            gloss_element.set('class', gloss)
                            word.append(gloss_element)
                            # continue
                    
                    # set remaining columns
                    else:
                        analysis = etree.Element(analysisid) 
                        analysis.attrib['class'] = value
                    
                    word.append(analysis)    
                    
                line.append(word)
                
        corpus.append(doc[0])

    etree.indent(corpus, space="    ")
    tree = etree.ElementTree(corpus)
    tree.write(out_path, pretty_print=True, xml_declaration=True, encoding="utf-8")


write_xml(data, metadata, out_path="../data/zaebuc_written_sample.xml",sample=100)
    # p(corpus)

/var/folders/nb/rn_wr53j4mbcyl_kw5fw8t3m0000gn/T/ipykernel_57812/764414433.py:29: PerformanceWarning: indexing past lexsort depth may impact performance.
  words = data.loc[(doc_id,lineidx)][:]
/var/folders/nb/rn_wr53j4mbcyl_kw5fw8t3m0000gn/T/ipykernel_57812/764414433.py:29: PerformanceWarning: indexing past lexsort depth may impact performance.
  words = data.loc[(doc_id,lineidx)][:]
/var/folders/nb/rn_wr53j4mbcyl_kw5fw8t3m0000gn/T/ipykernel_57812/764414433.py:29: PerformanceWarning: indexing past lexsort depth may impact performance.
  words = data.loc[(doc_id,lineidx)][:]
/var/folders/nb/rn_wr53j4mbcyl_kw5fw8t3m0000gn/T/ipykernel_57812/764414433.py:29: PerformanceWarning: indexing past lexsort depth may impact performance.
  words = data.loc[(doc_id,lineidx)][:]
/var/folders/nb/rn_wr53j4mbcyl_kw5fw8t3m0000gn/T/ipykernel_57812/764414433.py:29: PerformanceWarning: indexing past lexsort depth may impact performance.
  words = data.loc[(doc_id,lineidx)][:]
/var/folders/nb/rn_wr53j4mbcyl

In [117]:
data

word flag auto_tokenization  \
doc_id         Line_Index idx                                        
en-2019-116710 1.0        1    Developments  NaN      Developments   
                          2              in  NaN                in   
                          3             the  NaN               the   
                          4             UAE  NaN               UAE   
                          5               ,  NaN                 ,   
...                                     ...  ...               ...   
ar-2021-X8221  1.0        213          تمسك  NaN              تمسك   
                          214       المجتمع  NaN           المجتمع   
                          215        ببعضهم  NaN          ب+بعض+هم   
                          216          ببعض  NaN             ب+بعض   
                          217             .  NaN                 .   

                                    auto_pos   auto_lemma manual_tokenization  \
doc_id         Line_Index idx                                                   
en-2019-116710 1.0        1             NOUN  development        Developments   
                          2              ADP           in                  in   
                          3              DET          the                 the   
                          4            PROPN          UAE                 UAE   
                          5            PUNCT            ,                   ,   
...                                      ...          ...                 ...   
ar-2021-X8221  1.0        213           NOUN         تمسك                تمسك   
                          214           NOUN        مجتمع             المجتمع   
                          215  ADP+NOUN+PRON          بعض            ب+بعض+هم   
                          216       ADP+NOUN          بعض               ب+بعض   
                          217          PUNCT            .                   .   

                                  manual_pos manual_lemma comment  \
doc_id         Line_Index idx                                       
en-2019-116710 1.0        1             NOUN  development     NaN   
                          2              ADP           in     NaN   
                          3              DET          the     NaN   
                          4            PROPN          UAE     NaN   
                          5            PUNCT            ,     NaN   
...                                      ...          ...     ...   
ar-2021-X8221  1.0        213           NOUN         تمسك     NaN   
                          214           NOUN        مجتمع     NaN   
                          215  ADP+NOUN+PRON          بعض     NaN   
                          216       ADP+NOUN          بعض     NaN   
                          217          PUNCT            .     NaN   

                              manual_diacritized_lemma  \
doc_id         Line_Index idx                            
en-2019-116710 1.0        1                        NaN   
                          2                        NaN   
                          3                        NaN   
                          4                        NaN   
                          5                        NaN   
...                                                ...   
ar-2021-X8221  1.0        213                 تَمَسُّك   
                          214                مُجْتَمَع   
                          215                    بَعْض   
                          216                    بَعْض   
                          217                        .   

                                                   gloss core_pgn pron_pgn  \
doc_id         Line_Index idx                                                
en-2019-116710 1.0        1                          NaN      NaN      NaN   
                          2                          NaN      NaN      NaN   
                          3                          NaN      NaN      NaN   
                          4    

In [ ]:
data[]

word flag auto_tokenization  \
doc_id         Line_Index idx                                     
ar-2019-119813 1.0        1      التواصل  NaN           التواصل   
                          2    الاجتماعي  NaN         الاجتماعي   
                          3           له  NaN               ل+ه   
                          4       سلبيات  NaN            سلبيات   
                          5    وإيجابيات  NaN        و+إيجابيات   
...                                  ...  ...               ...   
ar-2021-X8221  1.0        213       تمسك  NaN              تمسك   
                          214    المجتمع  NaN           المجتمع   
                          215     ببعضهم  NaN          ب+بعض+هم   
                          216       ببعض  NaN             ب+بعض   
                          217          .  NaN                 .   

                                    auto_pos auto_lemma manual_tokenization  \
doc_id         Line_Index idx                                                 
ar-2019-119813 1.0        1             NOUN      تواصل             التواصل   
                          2              ADJ    اجتماعي           الاجتماعي   
                          3         ADP+PRON          ل                 ل+ه   
                          4             NOUN       سلبي              سلبيات   
                          5       CCONJ+NOUN     إيجابي          و+إيجابيات   
...                                      ...        ...                 ...   
ar-2021-X8221  1.0        213           NOUN       تمسك                تمسك   
                          214           NOUN      مجتمع             المجتمع   
                          215  ADP+NOUN+PRON        بعض            ب+بعض+هم   
                          216       ADP+NOUN        بعض               ب+بعض   
                          217          PUNCT          .                   .   

                                  manual_pos manual_lemma comment  \
doc_id         Line_Index idx                                       
ar-2019-119813 1.0        1             NOUN        تواصل     NaN   
                          2              ADJ      اجتماعي     NaN   
                          3         ADP+PRON            ل     NaN   
                          4             NOUN        سلبية     NaN   
                          5       CCONJ+NOUN      إيجابية     NaN   
...                                      ...          ...     ...   
ar-2021-X8221  1.0        213           NOUN         تمسك     NaN   
                          214           NOUN        مجتمع     NaN   
                          215  ADP+NOUN+PRON          بعض     NaN   
                          216       ADP+NOUN          بعض     NaN   
                          217          PUNCT            .     NaN   

                              manual_diacritized_lemma  \
doc_id         Line_Index idx                            
ar-2019-119813 1.0        1                    تَواصُل   
                          2               اِجْتِماعِيّ   
                          3                         لِ   
                          4                 سَلْبِيَّة   
                          5                إِيجابِيَّة   
...                                                ...   
ar-2021-X8221  1.0        213                 تَمَسُّك   
                          214                مُجْتَمَع   
                          215                    بَعْض   
                          216                    بَعْض   
                          217                        .   

                                                            gloss core_pgn  \
doc_id         Line_Index idx                                                
ar-2019-119813 1.0        1              continuation; continuity      ###   
                          2                                social      ###   
                          3                                for/to      ###   
                          4                            negativism      ###   
               

In [ ]:
l = data.loc[data.index[13]]['manual_lemma']
l
# data.loc[data.index[8]]

def get_translation_lemmas(english_lemma): 
    data.loc[data['gloss_search'].map(lambda x: english_lemma.lower() in list(x),na_action='ignore').where(lambda x: x==True).dropna().index,'manual_lemma'].drop_duplicates().to_list()
    return 


get_translation_lemmas(l)

['طمأنينة']

In [124]:
isar = data.index.map(lambda x: 'ar' in x[0])

data.loc[isar,'gloss_search'] = data['gloss'].map(lambda x: [x.strip().lower() for x in re.split(r',|;',x)],na_action='ignore')
data.loc[isar].apply(lambda x: x['gloss_search'].append(x['manual_lemma']),axis=1)

data.loc[~isar,'gloss_search'] = data.loc[~isar,'manual_lemma'].apply(get_translation_lemmas)

KeyboardInterrupt: 

'calm'

In [22]:
data.loc[('ar-2019-97236',   2)][:60]

/var/folders/nb/rn_wr53j4mbcyl_kw5fw8t3m0000gn/T/ipykernel_57812/4044103392.py:1: PerformanceWarning: indexing past lexsort depth may impact performance.
  data.loc[('ar-2019-97236',   2)][:60]


,word,flag,auto_tokenization,auto_pos,auto_lemma,manual_tokenization,manual_pos,manual_lemma,comment,manual_diacritized_lemma,gloss,core_pgn,pron_pgn
idx,,,,,,,,,,,,,
3,كلنا,NaN,كل+نا,NOUN+PRON,كل,كل+نا,NOUN+PRON,كل,NaN,كُلّ,all; every; entire; whole,###,1p
4,نعلم,NaN,نعلم,VERB,علم,نعلم,VERB,علم,NaN,عَلِم,be found out; be known; find out; be aware; know,1p,###
5,إلى,NaN,إلى,ADP,إلى,إلى,ADP,إلى,NaN,إِلَى,to; towards,###,###
6,أين,NaN,أين,ADV,أين,أين,ADV,أين,NaN,أَيْنَ,where,###,###
7,وصلت,NaN,وصلت,VERB,وصل,وصلت,VERB,وصل,NaN,وَصَل,reach; arrive at; be connected; be reached; co...,3fs,###
8,المرحلة,NaN,المرحلة,NOUN,مرحلة,المرحلة,NOUN,مرحلة,NaN,مَرْحَلَة,phase; phases; stages; rounds; stage; round,###,###
9,التطورية,NaN,التطورية,ADJ,تطوري,التطورية,ADJ,تطوري,NaN,تَطَوُّرِيّ,developmental; evolutionary,###,###
10,التي,NaN,التي,PRON,الذي,التي,PRON,التي,NaN,الَّتِي,which; who [fem.sg.]; who [fem.pl.]; whom [fem...,###,###
11,تسمى,NaN,تسمى,VERB,سمى,تسمى,VERB,سمى,NaN,سَمَّى,name; be designated; be called; be named; desi...,3fs,###


In [ ]:

metadata.columns.to_list()

['text',
 'language',
 'writer_id',
 'year',
 'course',
 'word_count',
 'cefr_avg',
 'major',
 'school_type',
 'school_language',
 'gender',
 'topic',
 'college',
 'residence',
 'writing_mins',
 'handwritten',
 'earlier_task_language',
 'days_between_tasks',
 'safe_assign_score',
 'cefr_1',
 'cefr_2',
 'cefr_3',
 'split']

In [359]:
data.loc[(doc_id, lineidx)][:]

/var/folders/nb/rn_wr53j4mbcyl_kw5fw8t3m0000gn/T/ipykernel_13595/1918117432.py:1: PerformanceWarning: indexing past lexsort depth may impact performance.
  data.loc[(doc_id, lineidx)][:]


,word,flag,auto_tokenization,auto_pos,auto_lemma,manual_tokenization,manual_pos,manual_lemma,comment,manual_diacritized_lemma,gloss,core_pgn,pron_pgn
idx,,,,,,,,,,,,,
3,تعتبر,NaN,تعتبر,VERB,اعتبر,تعتبر,VERB,اعتبر,NaN,اِعْتَبَر,believe; be regarded; consider; be considered;...,3fs,###
4,ثقافة,NaN,ثقافة,NOUN,ثقافة,ثقافة,NOUN,ثقافة,NaN,ثَقافَة,culture; civilization,###,###
5,التسامح,NaN,التسامح,NOUN,تسامح,التسامح,NOUN,تسامح,NaN,تَسامُح,tolerance,###,###
6,من,NaN,من,ADP,من,من,ADP,من,NaN,مِن,from,###,###
7,أهم,NaN,أهم,ADJ,أهم,أهم,ADJ,أهم,NaN,أَهَمّ,more/most important,###,###
...,...,...,...,...,...,...,...,...,...,...,...,...,...
143,ويسامح,NaN,و+يسامح,CCONJ+VERB,سامح,و+يسامح,CCONJ+VERB,سامح,NaN,سامَح,treat kindly; pardon; forgive,3ms,###
144,ويعفو,NaN,و+يعفو,CCONJ+VERB,عفا,و+يعفو,CCONJ+VERB,عفا,NaN,عَفا,be excused; excuse; be forgiven; forgive,3ms,###
145,عند,NaN,عند,NOUN,عند,عند,NOUN,عند,NaN,عِنْد,with/at,###,###


In [302]:
import lxml.etree.ElementTree

ModuleNotFoundError: No module named 'lxml.etree.ElementTree'; 'lxml.etree' is not a package

In [ ]:
import lxml.etree.ElementTree as ET

# ET.register_namespace(namespaces)
etree.register_namespace(namespaces)

from io import BytesIO

f = BytesIO()

tree = ET.ElementTree(corpus)
tree.register_namspace("xml",'http://www.w3.org/XML/1998/namespace')
tree.write(f, encoding='utf-8', xml_declaration=True)


print(f.getvalue().decode('utf-8'))

ModuleNotFoundError: No module named 'lxml.etree.ElementTree'; 'lxml.etree' is not a package

In [299]:
ET.ElementTree(doc)

In [180]:
data[data.index.get_level_values(1)==2]

Word Flag  ... Core_PGN Pron_PGN
doc_id         Line_Index idx                ...                  
en-2019-116710 2.0        24       The  NaN  ...      NaN      NaN
                          25       UAE  NaN  ...      NaN      NaN
                          26       has  NaN  ...      NaN      NaN
                          27         a  NaN  ...      NaN      NaN
                          28    desert  NaN  ...      NaN      NaN
...                                ...  ...  ...      ...      ...
ar-2021-X32635 2.0        183   للتعدي  NaN  ...      ###      ###
                          184      على  NaN  ...      ###      ###
                          185  خصوصيات  NaN  ...      ###      ###
                          186    غيرهم  NaN  ...      ###      3mp
                          187        .  NaN  ...      ###      ###

[44155 rows x 13 columns]

In [290]:
def p(xml): 
    return print(str(etree.tostring(xml, pretty_print=True,encoding='utf-8',xml_declaration=True).decode('utf-8')))

In [283]:
p(words)

TypeError: tostring() got an unexpected keyword argument 'resolve_entities'